# Dance video → SMPL motion (Colab GPU)

Runs a GPU SMPL estimator (**ROMP**) over your dance video and exports per-frame SMPL
pose params as `smpl.json`. Download it, then locally run:

```bash
node tools/smpl-to-vrma.mjs smpl.json public/vrma/dance.vrma   # → VRMA (carries limb twist)
node tools/render-vrma-preview.mjs --vrma public/vrma/dance.vrma --vrm /avatars/default.vrm --out dance.mp4 --contact-sheet
```

**Runtime → set GPU:** Runtime ▸ Change runtime type ▸ Hardware accelerator = **GPU (T4)**.

Why GPU/cloud: the M4 Mac has no CUDA; WHAM/GVHMR need it (DPVO). ROMP gives full
SMPL rotations (the *twist* the local MediaPipe pipeline can't). For even better quality
see the **Advanced** cell (WHAM/GVHMR) at the bottom.

⚠️ Licensing: only process video you have the right to use; keep outputs local.
Note: this notebook is provided ready-to-run but hasn't been executed on our side —
if a cell errors, paste the error back and we'll adjust.

In [ ]:
# 1) Install ROMP (SMPL-from-video) + deps
!pip -q install simple-romp opencv-python-headless 2>&1 | tail -3
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2) Upload your dance video (mp4/mov/webm)
from google.colab import files
up = files.upload()
VIDEO_PATH = list(up.keys())[0]
print('uploaded:', VIDEO_PATH)

In [ ]:
# 3) Run ROMP per frame → per-frame SMPL thetas (24 joints x 3 axis-angle = 72) → smpl.json
import json, cv2, numpy as np, romp

settings = romp.romp_settings(['--mode=image', '--GPU=0'])  # first run downloads the model (~hundreds MB)
model = romp.ROMP(settings)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
frames, last, idx, det = [], None, 0, 0
while True:
    ok, img = cap.read()
    if not ok: break
    out = None
    try: out = model(img)
    except Exception: out = None
    if out is not None and out.get('smpl_thetas') is not None and len(out['smpl_thetas']) > 0:
        p = 0
        j2 = out.get('pj2d_org')
        if j2 is not None and len(j2) > 1:  # multiple people → pick the largest (the dancer)
            areas = [ (k[:,0].max()-k[:,0].min())*(k[:,1].max()-k[:,1].min()) for k in j2 ]
            p = int(np.argmax(areas))
        thetas = np.asarray(out['smpl_thetas'][p]).reshape(-1).tolist()[:72]
        trans = (np.asarray(out['cam_trans'][p]).reshape(-1).tolist() if out.get('cam_trans') is not None else [0,0,0])
        last = {'thetas': thetas, 'trans': trans}; det += 1
    frames.append(last if last is not None else {'thetas':[0.0]*72, 'trans':[0,0,0]})
    idx += 1
    if idx % 30 == 0: print('frame', idx, 'detected', det)
cap.release()

json.dump({'fps': round(float(fps)), 'frames': frames, 'detected': det, 'source': 'romp'}, open('smpl.json','w'))
print('DONE:', len(frames), 'frames,', det, 'detected -> smpl.json')

In [ ]:
# 4) Download smpl.json
from google.colab import files
files.download('smpl.json')

## Advanced — WHAM / GVHMR (higher quality, world-grounded, anti foot-slide)

ROMP is per-frame (can jitter). For best quality use **GVHMR** or **WHAM** (both need this
GPU runtime). They are heavier to install (DPVO etc.); follow their repos and export per-frame
SMPL pose params into the SAME `smpl.json` shape (`{fps, frames:[{thetas:[72], trans:[3]}]}`),
then use the same local `smpl-to-vrma.mjs`.

- GVHMR: https://github.com/zju3dv/GVHMR
- WHAM:  https://github.com/yohanshin/WHAM

Tip: WHAM/GVHMR output SMPL in `(N, 24, 3)` or `(N, 72)` — flatten per frame to 72 and write `thetas`.